In [1]:
import pandas as pd
import os

# ✅ Place the modified engineer_date_features() here
def engineer_date_features(df, date_col):
    """
    Engineer date features without creating columns that produce NaN values.
    Skips: date_days_since, date_day, date_dayofweek, date_weekofyear, 
           date_quarter, date_month, date_year
    """
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    # Only keep safe boolean flags
    df["date_is_month_end"] = df[date_col].dt.is_month_end.astype(int)
    df["date_is_month_start"] = df[date_col].dt.is_month_start.astype(int)
    df["date_is_weekend"] = df[date_col].dt.weekday.isin([5, 6]).astype(int)

    # Drop the original date column if not needed
    df.drop(columns=[date_col], inplace=True, errors="ignore")
    
    return df

# Base path to your files
base_path = r"D:\ML\ML_projects\CreditPathAI\Microsoft_Loan_Data"

# Files to read
files = {
    "borrower": "Borrower.txt",
    "borrower_prod": "Borrower_Prod.txt",
    "loan": "Loan.txt",
    "loan_prod": "Loan_Prod.txt"
}

# Read files into dictionary of DataFrames
data = {}
for key, filename in files.items():
    filepath = os.path.join(base_path, filename)
    try:
        data[key] = pd.read_csv(filepath, sep='\t', low_memory=False)
        print(f"{key} loaded: {data[key].shape[0]} rows, {data[key].shape[1]} cols")
    except Exception as e:
        print(f"Error loading {filename}: {e}")

# Access like:
df_borrower = data["borrower"]
df_loan = data["loan"]

# Quick preview
df_borrower.head(), df_loan.head()


borrower loaded: 100000 rows, 17 cols
borrower_prod loaded: 22 rows, 17 cols
loan loaded: 100000 rows, 11 cols
loan_prod loaded: 22 rows, 11 cols


(   memberId residentialState yearsEmployment homeOwnership  annualIncome  \
 0   2305095               NM       10+ years          rent         56471   
 1   2610493               WA       2-5 years          rent         55038   
 2   2491679               MS        < 1 year          rent         56610   
 3   2092798               TX       6-9 years           own         54887   
 4   2633077               MA       2-5 years          rent         53522   
 
    incomeVerified  dtiRatio  lengthCreditHistory  numTotalCreditLines  \
 0               1     16.80                    6                   11   
 1               0     19.99                   22                    8   
 2               1     14.33                    5                    8   
 3               1     14.80                   12                   14   
 4               1     10.14                    4                   21   
 
    numOpenCreditLines  numOpenCreditLines1Year  revolvingBalance  \
 0                 9.

In [2]:
# ✅ Apply safe date feature engineering
df_loan = engineer_date_features(df_loan, "date")

print("\nDate features engineered safely. Columns now are:")
print(df_loan.columns)
print(df_loan.head())



Date features engineered safely. Columns now are:
Index(['loanId', 'memberId', 'purpose', 'isJointApplication', 'loanAmount',
       'term', 'interestRate', 'monthlyPayment', 'grade', 'loanStatus',
       'date_is_month_end', 'date_is_month_start', 'date_is_weekend'],
      dtype='object')
    loanId  memberId            purpose  isJointApplication  loanAmount  \
0  1888978   2305095  debtconsolidation                 0.0     25190.0   
1  1299695   2610493  debtconsolidation                 0.0     21189.0   
2  1875016   2491679  debtconsolidation                 0.0     29908.0   
3  1440478   2092798    homeimprovement                 0.0     13053.0   
4  1124634   2633077  debtconsolidation                 0.0     24613.0   

        term  interestRate  monthlyPayment grade loanStatus  \
0  60 months          6.25             490    E3    Current   
1  60 months         10.49             455    B3    Current   
2  60 months          9.11             622    B2    Current   
3  48

In [6]:
# --- Step 1: Define target and features ---
target_col = "loanStatus"  # Our y
y = df_loan[target_col]
X = df_loan.drop(columns=[target_col, "loanId", "memberId"], errors="ignore")  # drop IDs, target

print("✅ Features (X) and Target (y) separated")
print(f"X shape: {X.shape}, y shape: {y.shape}\n")

# --- Step 2: Check missing values ---
missing = X.isnull().sum()
missing_cols = missing[missing > 0]
print("Columns with missing values:\n", missing_cols if not missing_cols.empty else "No missing values!")

# --- Step 3: Quick look at target distribution ---
print("\nTarget distribution:")
print(y.value_counts(normalize=True))


✅ Features (X) and Target (y) separated
X shape: (100000, 10), y shape: (100000,)

Columns with missing values:
 isJointApplication     971
loanAmount            1006
term                  1071
dtype: int64

Target distribution:
loanStatus
Current    0.89996
Default    0.10004
Name: proportion, dtype: float64


In [7]:
# --- Step 1: Ordinal encoding ---
# Term: extract number of months
if "term" in df_loan.columns:
    df_loan["term"] = df_loan["term"].str.extract(r"(\d+)").astype(float)

# Years of employment: map to numeric
employment_map = {
    '< 1 year': 0.5,
    '1 year': 1,
    '2-5 years': 3.5,
    '6-9 years': 7.5,
    '10+ years': 10
}
if "yearsEmployment" in df_loan.columns:
    df_loan["yearsEmployment"] = df_loan["yearsEmployment"].map(employment_map)

# Grade: ordinal mapping
grade_map = {f"{l}{n}": i for i, (l, n) in enumerate(
    [(l, n) for l in ["A","B","C","D","E"] for n in range(1,4)]
)}
if "grade" in df_loan.columns:
    df_loan["grade"] = df_loan["grade"].map(grade_map)

print("✅ Ordinal encoding done.")
print(df_loan[["term","yearsEmployment","grade"]].head())

# --- Step 2: One-Hot Encoding for nominal columns ---
from sklearn.preprocessing import OneHotEncoder

nominal_cols = ["homeOwnership","purpose","residentialState"]
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# Fit-transform on df_loan
ohe_array = ohe.fit_transform(df_loan[nominal_cols])
ohe_cols = ohe.get_feature_names_out(nominal_cols)

# Convert to DataFrame and concatenate
df_ohe = pd.DataFrame(ohe_array, columns=ohe_cols, index=df_loan.index)
df_loan = pd.concat([df_loan.drop(columns=nominal_cols), df_ohe], axis=1)

print("✅ One-Hot Encoding done. New shape:", df_loan.shape)


✅ Ordinal encoding done.


KeyError: "['yearsEmployment'] not in index"

In [8]:
from sklearn.impute import SimpleImputer

# --- Impute numeric columns ---
num_cols = df_loan.select_dtypes(include=["number"]).columns
if len(num_cols) > 0:
    num_imputer = SimpleImputer(strategy="median")
    df_loan[num_cols] = num_imputer.fit_transform(df_loan[num_cols])

# --- Impute categorical columns (if any left) ---
cat_cols = df_loan.select_dtypes(exclude=["number"]).columns
if len(cat_cols) > 0:
    cat_imputer = SimpleImputer(strategy="most_frequent")
    df_loan[cat_cols] = cat_imputer.fit_transform(df_loan[cat_cols])

print("✅ Missing values handled (imputation complete)")


✅ Missing values handled (imputation complete)


In [17]:
from sklearn.model_selection import train_test_split

# --- Step 1: Define target and features ---
target_col = "loanStatus"
y = df_loan[target_col].map({"Current": 0, "Default": 1})  # encode target
X = df_loan.drop(columns=[target_col, "loanId", "memberId"], errors="ignore")  # drop IDs

# --- Step 2: Train-test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✅ Train shape: {X_train.shape}, Test shape: {X_test.shape}")


✅ Train shape: (80000, 10), Test shape: (20000, 10)


In [18]:
# Ensure numeric columns are aligned between train and test
numeric_cols = X_train.select_dtypes(include=["number"]).columns

X_train_numeric = X_train[numeric_cols].copy()
X_test_numeric = X_test[numeric_cols].copy()

print("✅ Numeric columns selected for scaling/SMOTE")



✅ Numeric columns selected for scaling/SMOTE


In [21]:
if "date" in df_loan.columns:
    df_loan = engineer_date_features(df, "date")
else:
    print("⚠️ Column 'date' not found, skipping date feature engineering")


⚠️ Column 'date' not found, skipping date feature engineering


In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier

# -----------------------------
# Step 0: Prepare your merged df_loan
# -----------------------------
# df_loan is already loaded and preprocessed

# -----------------------------
# Step 1: Split features and target
# -----------------------------
target_col = "loanStatus"
X = df_loan.drop(columns=[target_col])
y = df_loan[target_col].map({"Current": 0, "Default": 1})  # encode target

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

# -----------------------------
# Step 2: Keep only numeric features
# -----------------------------
num_cols = X_train.select_dtypes(include=["number"]).columns
X_train_num = X_train[num_cols].copy()
X_test_num = X_test[num_cols].copy()
print(f"Numeric features selected: {len(num_cols)}")

# -----------------------------
# Step 3: Scale numeric features
# -----------------------------
scaler = StandardScaler()
X_train_num[num_cols] = scaler.fit_transform(X_train_num[num_cols])
X_test_num[num_cols] = scaler.transform(X_test_num[num_cols])
print("✅ Numeric features scaled")

# -----------------------------
# Step 4: Apply SMOTE for class imbalance
# -----------------------------
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_num, y_train)
print(f"After SMOTE: X_train: {X_train_resampled.shape}, y_train: {y_train_resampled.shape}")

# -----------------------------
# Step 5: Train and evaluate models
# -----------------------------
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight="balanced"),
    "XGBoost": XGBClassifier(random_state=42, use_label_encoder=False, eval_metric="logloss"),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

for name, model in models.items():
    print(f"\n--- Training and Evaluating {name} ---")
    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test_num)

    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("Classification Report:")
    print(classification_report(y_test, y_pred))


Train shape: (80000, 12), Test shape: (20000, 12)
Numeric features selected: 11
✅ Numeric features scaled
After SMOTE: X_train: (143994, 11), y_train: (143994,)

--- Training and Evaluating Logistic Regression ---
Accuracy: 0.7635
Confusion Matrix:
[[13776  4223]
 [  507  1494]]
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.77      0.85     17999
           1       0.26      0.75      0.39      2001

    accuracy                           0.76     20000
   macro avg       0.61      0.76      0.62     20000
weighted avg       0.89      0.76      0.81     20000


--- Training and Evaluating Random Forest ---
Accuracy: 0.8828
Confusion Matrix:
[[16829  1170]
 [ 1175   826]]
Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.93      0.93     17999
           1       0.41      0.41      0.41      2001

    accuracy                           0.88     20000
   macro avg    

C:\Users\pingt\anacondanav\Lib\site-packages\xgboost\core.py:158: UserWarning: [19:55:35] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Accuracy: 0.8758
Confusion Matrix:
[[16600  1399]
 [ 1085   916]]
Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.92      0.93     17999
           1       0.40      0.46      0.42      2001

    accuracy                           0.88     20000
   macro avg       0.67      0.69      0.68     20000
weighted avg       0.88      0.88      0.88     20000


--- Training and Evaluating Gradient Boosting ---
Accuracy: 0.8406
Confusion Matrix:
[[15573  2426]
 [  763  1238]]
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.87      0.91     17999
           1       0.34      0.62      0.44      2001

    accuracy                           0.84     20000
   macro avg       0.65      0.74      0.67     20000
weighted avg       0.89      0.84      0.86     20000


--- Training and Evaluating AdaBoost ---


C:\Users\pingt\anacondanav\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Accuracy: 0.7929
Confusion Matrix:
[[14464  3535]
 [  607  1394]]
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.80      0.87     17999
           1       0.28      0.70      0.40      2001

    accuracy                           0.79     20000
   macro avg       0.62      0.75      0.64     20000
weighted avg       0.89      0.79      0.83     20000



In [23]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier

# Dictionary to store results
results = []

# List of models (you already have them trained/fitted)
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight="balanced"),
    "XGBoost": XGBClassifier(random_state=42, use_label_encoder=False, eval_metric="logloss"),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

# Train, predict, and store metrics in a DataFrame
for name, model in models.items():
    model.fit(X_train_resampled, y_train_resampled)  # SMOTE-applied training set from df_loan
    y_pred = model.predict(X_test_num)               # Numeric test features from df_loan
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (0)": precision_score(y_test, y_pred, pos_label=0),
        "Precision (1)": precision_score(y_test, y_pred, pos_label=1),
        "Recall (0)": recall_score(y_test, y_pred, pos_label=0),
        "Recall (1)": recall_score(y_test, y_pred, pos_label=1),
        "F1 (0)": f1_score(y_test, y_pred, pos_label=0),
        "F1 (1)": f1_score(y_test, y_pred, pos_label=1)
    })

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Format numbers for readability
metrics_cols = ["Accuracy", "Precision (0)", "Precision (1)", "Recall (0)", "Recall (1)", "F1 (0)", "F1 (1)"]
results_df[metrics_cols] = results_df[metrics_cols].round(4)

# Display
display(results_df)


C:\Users\pingt\anacondanav\Lib\site-packages\xgboost\core.py:158: UserWarning: [19:58:48] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\pingt\anacondanav\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


,Model,Accuracy,Precision (0),Precision (1),Recall (0),Recall (1),F1 (0),F1 (1)
0,Logistic Regression,0.7635,0.9645,0.2613,0.7654,0.7466,0.8535,0.3871
1,Random Forest,0.8828,0.9347,0.4138,0.9350,0.4128,0.9349,0.4133
2,XGBoost,0.8758,0.9386,0.3957,0.9223,0.4578,0.9304,0.4245
3,Gradient Boosting,0.8406,0.9533,0.3379,0.8652,0.6187,0.9071,0.4371
4,AdaBoost,0.7929,0.9597,0.2828,0.8036,0.6967,0.8748,0.4023
